In [2]:
import pandas as pd
df=pd.read_csv("../data/raw/train.csv")
print(df.shape)
df.columns.tolist()

(9800, 18)


['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales']

In [3]:
df.dtypes

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
dtype: object

In [5]:
print("Duplicate rows:", df.duplicated)
df.isnull().sum()

Duplicate rows: <bound method DataFrame.duplicated of       Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
0          1  CA-2017-152156  08/11/2017  11/11/2017    Second Class   
1          2  CA-2017-152156  08/11/2017  11/11/2017    Second Class   
2          3  CA-2017-138688  12/06/2017  16/06/2017    Second Class   
3          4  US-2016-108966  11/10/2016  18/10/2016  Standard Class   
4          5  US-2016-108966  11/10/2016  18/10/2016  Standard Class   
...      ...             ...         ...         ...             ...   
9795    9796  CA-2017-125920  21/05/2017  28/05/2017  Standard Class   
9796    9797  CA-2016-128608  12/01/2016  17/01/2016  Standard Class   
9797    9798  CA-2016-128608  12/01/2016  17/01/2016  Standard Class   
9798    9799  CA-2016-128608  12/01/2016  17/01/2016  Standard Class   
9799    9800  CA-2016-128608  12/01/2016  17/01/2016  Standard Class   

     Customer ID     Customer Name    Segment        Country             City  \


Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

In [8]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True)
df[["Order Date", "Ship Date"]].dtypes

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [12]:
#remove duplicate rows
df=df.drop_duplicates()

#fill misssing postal codes(only matters if step4 showed misssing values, because postal codes have 11 missing values)
if "Postal Code" in df.columns:
    df["Postal Code"] = df["Postal Code"].fillna(0).astype(int).astype(str)

#remove extra spaces in text columns
for col in df.select_dtypes(include="object").columns:
    df[col]=df[col].str.strip()

print(df.shape)
df.isnull().sum().sum()

(9800, 18)


C:\Users\ravin\AppData\Local\Temp\ipykernel_8732\1281152229.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


np.int64(0)

In [13]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)
df[["Order Date", "Year", "Month", "YearMonth"]].head()

,Order Date,Year,Month,YearMonth
0,2017-11-08,2017,11,2017-11
1,2017-11-08,2017,11,2017-11
2,2017-06-12,2017,6,2017-06
3,2016-10-11,2016,10,2016-10
4,2016-10-11,2016,10,2016-10


In [14]:
print("Date range:", df["Order Date"].min(), "to", df["Order Date"].max())
print("Negative or zero sales:", (df["Sales"]<=0).sum())
print("New shape:", df.shape)
df["Sales"].describe()

Date range: 2015-01-03 00:00:00 to 2018-12-30 00:00:00
Negative or zero sales: 0
New shape: (9800, 21)


count     9800.000000
mean       230.769059
std        626.651875
min          0.444000
25%         17.248000
50%         54.490000
75%        210.605000
max      22638.480000
Name: Sales, dtype: float64

In [15]:
df.to_csv("../data/processed/clean_sales.csv", index=False)

check = pd.read_csv("../data/processed/clean_sales.csv", parse_dates=["Order Date", "Ship Date"])
print(check.shape)
check.dtypes.head(8)

(9800, 21)


Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
dtype: object